In [1]:
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import numpy as np
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from pandas.tseries.offsets import BDay
import matplotlib.pyplot as plt


In [2]:
def createData(stocks):
    data=[]
    for i,stock in enumerate(stocks):
        stock_history = yf.download(stock, period='12mo',interval='1d')
        stock_history[('SMA_50',stock)] = stock_history['Close'].rolling(window=50).mean()
        stock_history[('SMA_200',stock)] = stock_history['Close'].rolling(window=200).mean()
        def compute_rsi(series, period=14):
            delta = series.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)

            avg_gain = gain.rolling(window=period).mean()
            avg_loss = loss.rolling(window=period).mean()

            rs = avg_gain / avg_loss
            rsi = 100 - (100 / (1 + rs))
            return rsi

        stock_history[('RSI',stock)] = compute_rsi(stock_history['Close'])
        ema_12 = stock_history['Close'].ewm(span=12, adjust=False).mean()
        ema_26 = stock_history['Close'].ewm(span=26, adjust=False).mean()
        stock_history[('MACD',stock)] = ema_12 - ema_26
        stock_history[('MACD_Signal',stock)] = stock_history['MACD'].ewm(span=9, adjust=False).mean()
        ma_20 = stock_history['Close'].rolling(window=20).mean()
        std_20 = stock_history['Close'].rolling(window=20).std()
        # stock_history[('Bollinger_Upper',stock)] = ma_20 + (2 * std_20)
        # stock_history[('Bollinger_Lower',stock)] = ma_20 - (2 * std_20)
        low_14 = stock_history['Low'].rolling(window=14).min()
        high_14 = stock_history['High'].rolling(window=14).max()
        stock_history[('%K',stock)] = 100 * ((stock_history['Close'] - low_14) / (high_14 - low_14))
        stock_history[('%D',stock)] = stock_history['%K'].rolling(window=3).mean()
        stock_history[('Volatility_14',stock)] = stock_history['Close'].rolling(window=14).std()
        stock_history = stock_history.stack(level=1).reset_index()
        for lag in [1,2,3]:
            stock_history[f'Close_lag_{lag}'] = stock_history.groupby('Ticker')['Close'].shift(lag)
            stock_history[f'Volume_lag_{lag}'] = stock_history.groupby('Ticker')['Volume'].shift(lag)

        # ==== Rolling Statistics ====
        for window in [5,20]:
            stock_history[f'RollingMean_{window}'] = stock_history.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window).mean())
            stock_history[f'RollingStd_{window}'] = stock_history.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window).std())
            stock_history[f'RollingVol_{window}'] = stock_history.groupby('Ticker')['Volume'].transform(lambda x: x.rolling(window).mean())

        # ==== Log Returns ====
        # stock_history['LogReturn_1'] = stock_history.groupby('Ticker')['Close'].transform(lambda x: np.log(x / x.shift(1)))
        # stock_history['LogReturn_5'] = stock_history.groupby('Ticker')['Close'].transform(lambda x: np.log(x / x.shift(5)))

        # ==== Relative Volume ====
        # stock_history['RelVolume_20'] = stock_history['Volume'] / (stock_history.groupby('Ticker')['Volume'].transform(lambda x: x.rolling(20).mean()))

        # ==== Seasonality Features ====
        # stock_history['DayOfWeek'] = stock_history['Date'].dt.dayofweek
        stock_history['Ticker']=stock
        stock_history['Target_7d'] = stock_history['Close'].shift(-5) # named 7 days as there will be no saturday and sunday in dataset but for same day next week the date counts
        stock_history['LogReturn_5d'] = np.log(stock_history['Target_7d'] / stock_history['Close'])
        stock_history['sentiment_score']=0
        stock_history.dropna(inplace=True)
        data.append(stock_history)


    df = pd.concat(data)
    df.reset_index()
    return df




In [3]:
def compute_rsi(close_series, period=14):
    delta = close_series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


In [4]:
def make_prediction_sequences30(df, feature_cols, time_steps=30):
    X_seq, X_ticker = [], []
    
    for i in range(len(df) - time_steps):
        seq = df.iloc[i:i+time_steps][feature_cols].values
        X_seq.append(seq)
    
    return np.array(X_seq)


In [5]:
def squeeze_axis_1(x):
    return tf.squeeze(x, axis=1)


model = load_model(
    "../stock_prediction_backend/models/stock_model_multi_US_2022_sentiment_score.keras",
    custom_objects={'squeeze_axis_1': squeeze_axis_1},
    safe_mode=False
)

In [6]:
df = createData(["AAPL"])
current_date = datetime.now()

# Get last 35 trading days (approximately 7 weeks)
test_data = df.tail(35)

time_steps = 30
if len(test_data) < time_steps:
    raise ValueError("Not enough data points for the specified time_steps")

X_series_scaler = joblib.load("../stock_prediction_backend/models/X_series_scaler_multistock_US_New1.joblib")
y_scaler = joblib.load("../stock_prediction_backend/models/stock_model_orcl_aapl_amzn_googl2022_New1.joblib")  
ticker_id = 0

predictions = []
prediction_dates = []

# Run prediction 5 times
for prediction_count in range(5):
    sequence = test_data.iloc[-time_steps:]
    if sequence['Ticker'].nunique() > 1:
        continue
    
    # Prepare sequence for model
    new_sequence = sequence.drop(['Date', 'Ticker','Target_7d','LogReturn_5d'], axis=1)
    sequence_2d = np.array(new_sequence).reshape(-1, new_sequence.shape[-1])
    sequence_scaled = X_series_scaler.transform(sequence_2d)
    sequence_scaled = sequence_scaled.reshape(1, time_steps, new_sequence.shape[-1])

    # Make prediction
    ticker_input = np.array([[ticker_id]])
    y_pred_scaled = model.predict([ticker_input, sequence_scaled])
    y_pred_log_return = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()[0]

    # Calculate predicted price
    current_close = test_data.iloc[-1]['Close']
    predicted_close = current_close * np.exp(y_pred_log_return)
    
    # Calculate next trading day
    last_date = test_data.iloc[-1]['Date']
    next_trading_day = last_date + BDay(1)
    # Store results
    predictions.append(predicted_close)
    prediction_dates.append(next_trading_day)

    
    print(f'Prediction {prediction_count + 1}: {next_trading_day.date()} --> {predicted_close:.2f}')
    
    # Append the prediction to test_data for next iteration
    new_row = test_data.iloc[-1].copy()
    new_row['Close'] = predicted_close
    new_row['Date'] = next_trading_day
    test_data = pd.concat([test_data, pd.DataFrame([new_row])], ignore_index=True)

# Create results DataFrame
results = pd.DataFrame({
    'PredictionDate': prediction_dates,
    'PredictedPrice': predictions
})

print("\nFinal Predictions:")
print(results)

C:\Users\dhruv\AppData\Local\Temp\ipykernel_32452\2905822006.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_history = yf.download(stock, period='12mo',interval='1d')
[*********************100%***********************]  1 of 1 completed
C:\Users\dhruv\AppData\Local\Temp\ipykernel_32452\2905822006.py:33: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stock_history = stock_history.stack(level=1).reset_index()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step
Prediction 1: 2025-07-21 --> 210.82
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction 2: 2025-07-22 --> 210.41
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction 3: 2025-07-23 --> 209.96
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Prediction 4: 2025-07-24 --> 209.47
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Prediction 5: 2025-07-25 --> 208.96

Final Predictions:
  PredictionDate  PredictedPrice
0     2025-07-21      210.815099
1     2025-07-22      210.406631
2     2025-07-23      209.958207
3     2025-07-24      209.473834
4     2025-07-25      208.957754
